# 07 — Vector Spaces, Span, Basis, Rank & Null Space
## Understanding the Structure of Your Data

> **Prerequisites:** Notebooks 01–06  
> **Difficulty:** ⭐⭐⭐☆☆ Intermediate  
> **Running example:** Feature analysis of house data

---

## Why does this matter for ML?

Have you ever had a dataset where adding more features didn't improve your model?
Have you ever wondered why some linear regression models become unstable?
Do you want to know what "dimensionality" of data really means?

All of these questions are answered by the concepts in this notebook:

| Concept | ML question it answers |
|---|---|
| **Linear independence** | Do my features contain redundant information? |
| **Rank** | How many truly independent dimensions does my data have? |
| **Span / Column space** | What outputs can my model actually produce? |
| **Null space** | What inputs does my model ignore? |
| **Basis** | What is the minimal description of my data? |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

---
## 1. Vector Space — A "Closed" World

### Plain English
A **vector space** is a collection of vectors where you can:
1. **Add** any two vectors and stay in the collection
2. **Multiply any vector by any scalar** and stay in the collection

It's a world that's "closed" — you can't escape by adding or scaling.

### Examples
- All 2D vectors ℝ² = every arrow you can draw on a plane (infinite)
- All 3D vectors ℝ³ = every arrow in 3D space
- A **subspace** is a smaller vector space living inside a bigger one

### What makes something a subspace?
1. It must contain the **zero vector** (the origin)
2. Closed under addition: u + v is in the subspace
3. Closed under scaling: c×u is in the subspace

### Why care in ML?
A neural network layer transforms data from one vector space to another. Understanding that each layer is a linear map between vector spaces helps you understand **what the network can and cannot learn**.

In [ ]:
# ─────────────────────────────────────────────────────────
# VECTOR SPACE: checking closure properties
# ─────────────────────────────────────────────────────────

print("=== Vector space closure ===")
print()

# ℝ² is a vector space — everything stays in ℝ²
v1 = np.array([2., 3.])
v2 = np.array([-1., 4.])
print(f"v1 = {v1}, v2 = {v2}  (both in ℝ²)")
print(f"v1 + v2 = {v1+v2}  (still in ℝ²? Yes)")
print(f"5 × v1  = {5*v1}  (still in ℝ²? Yes)")

print()
print("=== A plane through the origin IS a subspace of ℝ³ ===")
# Plane: x + y + z = 0
def on_plane(v):
    return abs(v[0] + v[1] + v[2]) < 1e-10

a = np.array([1., -2., 1.])   # 1-2+1=0 ✓
b = np.array([2.,  0., -2.])  # 2+0-2=0 ✓
print(f"a = {a}, on plane: {on_plane(a)}")
print(f"b = {b}, on plane: {on_plane(b)}")
print(f"a+b = {a+b}, on plane: {on_plane(a+b)}   ← stays in subspace!")
print(f"3*a = {3*a}, on plane: {on_plane(3*a)}  ← stays in subspace!")

print()
print("=== A plane NOT through origin is NOT a subspace ===")
# Plane: x + y + z = 1 (doesn't pass through origin)
def on_shifted_plane(v):
    return abs(v[0] + v[1] + v[2] - 1) < 1e-10

c = np.array([1., 0., 0.])   # on the shifted plane
d = np.array([0., 1., 0.])   # on the shifted plane
print(f"c = {c}, d = {d}")
print(f"c+d = {c+d}, on shifted plane: {on_shifted_plane(c+d)}  ← NOT in the set!")
print("(Because doesn't contain zero: [0,0,0] gives 0≠1)")

---
## 2. Linear Independence — No Redundancy

### Plain English
Vectors are **linearly independent** if you cannot make any one of them from a combination of the others.

Each independent vector adds **genuinely new information**.

### Test
Vectors v₁, v₂, ..., vₙ are independent if the **only** solution to:
> c₁v₁ + c₂v₂ + ... + cₙvₙ = 0
is c₁ = c₂ = ... = cₙ = 0 (all coefficients zero).

### Intuition
- [1, 0] and [0, 1] are independent: you can't make [1,0] from [0,1] or vice versa
- [1, 0] and [2, 0] are **dependent**: [2, 0] = 2 × [1, 0] — redundant!
- [1, 0], [0, 1], [1, 1] are **dependent**: [1,1] = [1,0] + [0,1] — the third is a combo of the first two

### ML connection: redundant features
If your features are linearly dependent (e.g., `price_USD` and `price_EUR = 1.1 × price_USD`), adding both to your model adds **no new information** and can cause numerical problems.

In [ ]:
# ─────────────────────────────────────────────────────────
# LINEAR INDEPENDENCE
# ─────────────────────────────────────────────────────────

print("=== Linear independence via matrix rank ===")
print("Stack vectors as rows/columns and compute rank.")
print("If rank < number of vectors → some are dependent.")
print()

def check_independence(*vectors, labels=None):
    M = np.column_stack(vectors)          # put vectors as columns
    rank = np.linalg.matrix_rank(M)
    n = len(vectors)
    independent = (rank == n)
    lbl = labels if labels else [f'v{i+1}' for i in range(n)]
    print(f"Vectors {lbl}: rank={rank}/{n}, independent={independent}")
    if not independent:
        print(f"  → {n-rank} vector(s) are redundant")
    return independent

check_independence(np.array([1.,0.]), np.array([0.,1.]), labels=['[1,0]','[0,1]'])
check_independence(np.array([1.,0.]), np.array([2.,0.]), labels=['[1,0]','[2,0]'])
check_independence(np.array([1.,0.,0.]), np.array([0.,1.,0.]), np.array([0.,0.,1.]),
                   labels=['e1','e2','e3'])
check_independence(np.array([1.,0.,0.]), np.array([0.,1.,0.]), np.array([1.,1.,0.]),
                   labels=['e1','e2','e1+e2'])

print()
print("=== ML example: detecting redundant features ===")
# Dataset with a redundant feature
house_sizes = np.array([1500., 2000., 1200., 1800., 2500.])
size_sqft  = house_sizes
size_sqm   = house_sizes * 0.0929   # 1 sqft = 0.0929 sqm (perfectly redundant!)
bedrooms   = np.array([3., 4., 2., 3., 5.])

X_redundant = np.column_stack([size_sqft, size_sqm, bedrooms])
rank = np.linalg.matrix_rank(X_redundant)
print(f"Features: [size_sqft, size_sqm, bedrooms]")
print(f"Rank: {rank} out of 3 features")
print(f"→ size_sqm = 0.0929 × size_sqft → completely redundant!")
print(f"→ Only {rank} truly independent features, not 3")

---
## 3. Span — What Can We Reach?

### Plain English
The **span** of a set of vectors is all the vectors you can make by adding and scaling them.

> span({v₁, v₂, ..., vₙ}) = all vectors of the form **c₁v₁ + c₂v₂ + ... + cₙvₙ**

### Examples
- span of [1, 0] alone → the entire x-axis (scale it to get any point on x)
- span of [1, 0] and [0, 1] → the entire ℝ² plane
- span of [1, 0] and [2, 0] → still just the x-axis! (dependent vectors don't add new reach)

### In ML
The **column span of A** = all possible outputs of Ax.
- If b is NOT in the column span of A, then Ax = b has **no solution**
- This is why having independent features matters!

In [ ]:
# ─────────────────────────────────────────────────────────
# SPAN: visualizing what combinations can reach
# ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Subplot 1: span of one vector = a line
ax = axes[0]
v = np.array([1., 2.])
t = np.linspace(-2.5, 2.5, 100)
line = np.outer(t, v)
ax.plot(line[:,0], line[:,1], 'steelblue', lw=2, label='span({v}) = a line')
ax.annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.text(1.1, 2.1, 'v=[1,2]', color='red', fontsize=11)
ax.set_xlim(-3,3); ax.set_ylim(-5,5)
ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
ax.grid(True, alpha=0.3); ax.set_aspect('equal')
ax.legend(fontsize=9); ax.set_title('span of 1 vector = LINE', fontsize=10)

# Subplot 2: span of 2 independent vectors = the plane
ax = axes[1]
v1 = np.array([2., 0.5]); v2 = np.array([0., 1.])
for c1 in np.linspace(-2, 2, 12):
    for c2 in np.linspace(-2, 2, 12):
        pt = c1*v1 + c2*v2
        ax.plot(pt[0], pt[1], 'b.', markersize=2, alpha=0.5)
ax.annotate('', xy=v1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.annotate('', xy=v2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='green', lw=2.5))
ax.set_xlim(-3,3); ax.set_ylim(-2.5,2.5)
ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
ax.grid(True, alpha=0.3); ax.set_aspect('equal')
ax.set_title('span of 2 independent vectors = PLANE ℝ²', fontsize=10)

# Subplot 3: span of 2 dependent vectors = still just a line
ax = axes[2]
v1 = np.array([1., 2.]); v2 = np.array([2., 4.])  # v2 = 2*v1
t = np.linspace(-2, 2, 100)
ax.plot(t, 2*t, 'steelblue', lw=2, label='span = still a line!')
ax.annotate('', xy=v1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='red', lw=2.5))
ax.annotate('', xy=v2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='orange', lw=2.5))
ax.text(1.1, 2.1, 'v1', color='red', fontsize=11)
ax.text(2.1, 4.1, 'v2=2v1', color='orange', fontsize=11)
ax.set_xlim(-3,3); ax.set_ylim(-5,5)
ax.axhline(0,'k',lw=0.5); ax.axvline(0,'k',lw=0.5)
ax.grid(True, alpha=0.3); ax.set_aspect('equal')
ax.legend(fontsize=9); ax.set_title('span of 2 DEPENDENT vectors = LINE\n(v2=2*v1 adds no new reach!)', fontsize=10)

plt.suptitle('Span = Everything You Can Reach via Linear Combinations', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Basis — The Minimal Description

### Plain English
A **basis** is the smallest set of vectors that can reach everything in a vector space:
- **Spans** the whole space (can reach every point)
- **Linearly independent** (no redundancy)

The number of vectors in any basis = the **dimension** of the space.

### Examples
- Standard basis of ℝ²: {[1,0], [0,1]} — the x and y unit vectors
- Alternative basis of ℝ²: {[1,1], [1,-1]} — rotated 45°, also valid!
- Any 3 independent vectors form a basis for ℝ³

### Why multiple bases?
Different bases are better for different purposes:
- **Standard basis**: good for general computation
- **PCA basis**: aligned with data directions — better for compression
- **Fourier basis**: aligned with frequencies — better for audio/images

In [ ]:
# ─────────────────────────────────────────────────────────
# BASIS: different coordinate systems
# ─────────────────────────────────────────────────────────

print("=== Basis: different ways to describe the same point ===")
print()

# Standard basis
e1 = np.array([1., 0.])   # unit x
e2 = np.array([0., 1.])   # unit y

# Alternative basis (rotated 45°)
b1 = np.array([1., 1.]) / np.sqrt(2)   # diagonal right-up
b2 = np.array([-1., 1.]) / np.sqrt(2)  # diagonal left-up

# A point in ℝ²
point = np.array([3., 2.])

# Express point in standard basis
print(f"Point: {point}")
print(f"In standard basis: {point[0]}×{e1} + {point[1]}×{e2}")
print(f"  Coordinates: ({point[0]}, {point[1]})")

# Express same point in the alternative basis
B = np.column_stack([b1, b2])  # columns are the new basis vectors
coords_new = np.linalg.solve(B, point)  # solve for coordinates in new basis
print(f"\nIn alternative basis (rotated 45°):")
print(f"  Coordinates: ({coords_new[0]:.4f}, {coords_new[1]:.4f})")
print(f"  Verify: {coords_new[0]:.4f}×{b1.round(4)} + {coords_new[1]:.4f}×{b2.round(4)}")
print(f"       = {(coords_new[0]*b1 + coords_new[1]*b2).round(4)}  (should be {point})")

print()
print("SAME POINT, different numbers — because different 'rulers'!")
print("This is change of basis — fundamental to PCA and feature engineering.")

---
## 5. Rank — The True Dimensionality

### Plain English
The **rank** of a matrix is:
- The number of **truly independent rows** (= independent columns — always the same!)
- How many dimensions the data actually lives in
- Number of non-zero singular values (from SVD)

### What rank tells you in ML
| Situation | What it means |
|---|---|
| rank(X) < n_features | Features are redundant — not all independent |
| rank(X) = n_features | Full rank — all features add unique information |
| rank(X) < n_samples | Normal for tall datasets (more samples than features) |

In [ ]:
# ─────────────────────────────────────────────────────────
# RANK
# ─────────────────────────────────────────────────────────

print("=== Computing rank ===")
print()

# Full rank: 2×2 matrix with independent rows
A = np.array([[1., 2.], [3., 4.]])
print(f"A = [[1,2],[3,4]], rank = {np.linalg.matrix_rank(A)} (full rank)")

# Rank deficient: row 2 = 3 × row 1
B = np.array([[1., 2.], [3., 6.]])
print(f"B = [[1,2],[3,6]], rank = {np.linalg.matrix_rank(B)} (rank 1: row2=3×row1)")

# Dataset example
print()
print("=== House dataset rank analysis ===")
X = np.array([
    [1500., 3., 10., 5.2],
    [2100., 4.,  5., 2.8],
    [1200., 2., 20., 8.0],
    [1800., 3.,  8., 4.5],
    [2500., 5.,  2., 1.0],
])
print(f"Dataset X shape: {X.shape}")
print(f"Rank: {np.linalg.matrix_rank(X)} (= min(5,4) = 4 → full column rank)")
print(f"→ All 4 features are independent — no redundancy")

# Add a redundant feature
X_bad = np.column_stack([X, X[:,0] * 2])  # add size_x2 = 2 × size
print(f"\nWith duplicate feature (size × 2) added, shape: {X_bad.shape}")
print(f"Rank: {np.linalg.matrix_rank(X_bad)} (still 4! The new column is redundant)")
print(f"→ 5 columns but only 4 independent directions")

---
## 6. Null Space — What Gets Ignored?

### Plain English
The **null space** of A is all the vectors x that A sends to zero:
> **Null(A) = { x such that Ax = 0 }**

### Intuition: the shadow analogy
Imagine shining a light on a fence — the fence casts a shadow (a projection). Any light ray that hits the fence **parallel** to the fence boards casts no shadow — it's "invisible" to the shadow. Those are in the null space of the shadow projection.

### The Rank-Nullity Theorem
For an m×n matrix A:
> **rank(A) + nullity(A) = n**

Where nullity = dimension of null space.

- If rank = n → null space = {0} only → unique solution to Ax=b
- If rank < n → non-trivial null space → infinite solutions to Ax=0

In [ ]:
# ─────────────────────────────────────────────────────────
# NULL SPACE
# ─────────────────────────────────────────────────────────

print("=== Null space ===")
print()

# A rank-deficient matrix has a non-trivial null space
A = np.array([[1., 2., 3.],
              [2., 4., 6.]])  # row 2 = 2×row 1, rank = 1

rank = np.linalg.matrix_rank(A)
nullity = A.shape[1] - rank

print(f"A:\n{A}")
print(f"Shape: {A.shape}, rank={rank}, nullity={nullity}")
print(f"rank + nullity = {rank} + {nullity} = {rank+nullity} = n={A.shape[1]} ✓")
print()

# Find null space via SVD (rows of Vt corresponding to near-zero singular values)
U, s, Vt = np.linalg.svd(A, full_matrices=False)
print(f"Singular values: {s.round(6)}")
print(f"Near-zero singular values: {(s < 1e-10).sum()} (that's the nullity)")
print()

# BUT full_matrices=False might not give us the full null space for rectangular A
# Use the full SVD instead
U, s_full, Vt_full = np.linalg.svd(A, full_matrices=True)
# Null space = last (n-rank) rows of Vt_full
null_space = Vt_full[rank:, :]  # rows beyond the rank
print(f"Null space basis vectors (one per row):")
print(null_space.round(4))
print()

# Verify each null vector satisfies A @ v = 0
for i, nv in enumerate(null_space):
    result = A @ nv
    print(f"A @ null_vec_{i+1} = {result.round(8)}  (should be [0,0]) ✓")

---
## Summary

| Concept | In plain words | NumPy |
|---|---|---|
| **Vector space** | Closed set of vectors | All of ℝⁿ, or a subspace |
| **Linear independence** | No vector is a combination of others | `np.linalg.matrix_rank` |
| **Span** | Everything reachable by combining vectors | Column space of a matrix |
| **Basis** | Smallest spanning independent set | Columns of U from SVD |
| **Rank** | # independent rows/columns = true dimensionality | `np.linalg.matrix_rank(A)` |
| **Null space** | All x with Ax=0 | Rows of Vt beyond the rank |
| **Rank-nullity** | rank + nullity = n | Always true! |

**Next: Notebook 08 — Dot Products, Norms & Distance Metrics** (how to measure similarity and length)